In [1]:

from pathlib import Path

# Create the project directory
PROJECT_DIR = Path("/content/resume-rag-project")

# Create all required folders
folders = [
    "resumes",
    "job_descriptions",
    "vector_db",
    "evaluation"
]

for folder in folders:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Project directory created successfully!")
print("Location:", PROJECT_DIR)
print("\nFolders created:")

for folder in folders:
    print("📁", folder)


Project directory created successfully!
Location: /content/resume-rag-project

Folders created:
📁 resumes
📁 job_descriptions
📁 vector_db
📁 evaluation


In [2]:

%pip install -q sentence-transformers chromadb rank-bm25 pypdf python-docx pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:

import sentence_transformers
import chromadb
import pandas
import sklearn
from rank_bm25 import BM25Okapi
from pypdf import PdfReader
from docx import Document

print("All required libraries imported successfully!")
print("Sentence Transformers:", sentence_transformers.__version__)
print("ChromaDB:", chromadb.__version__)


All required libraries imported successfully!
Sentence Transformers: 5.7.0
ChromaDB: 1.5.9


In [4]:

from pathlib import Path
import json
import random

random.seed(42)

RESUME_DIR = PROJECT_DIR / "resumes"
JD_DIR = PROJECT_DIR / "job_descriptions"

# Different profiles to create a diverse test dataset.
profiles = [
    ("Software Engineer", ["Python", "Java", "SQL", "Git"], "Backend development"),
    ("Frontend Developer", ["JavaScript", "React", "HTML", "CSS"], "Frontend development"),
    ("Full Stack Developer", ["Java", "Angular", "Spring Boot", "SQL"], "Full-stack applications"),
    ("Data Engineer", ["Python", "SQL", "Spark", "AWS"], "Data pipelines"),
    ("Machine Learning Engineer", ["Python", "scikit-learn", "TensorFlow", "SQL"], "Machine learning"),
    ("DevOps Engineer", ["AWS", "Docker", "Kubernetes", "Linux"], "Cloud infrastructure"),
    ("Java Developer", ["Java", "Spring Boot", "REST APIs", "SQL"], "Enterprise applications"),
    ("Python Developer", ["Python", "Django", "FastAPI", "PostgreSQL"], "API development"),
    ("Cloud Engineer", ["AWS", "Terraform", "Docker", "Linux"], "Cloud platforms"),
    ("QA Engineer", ["Python", "Selenium", "SQL", "Git"], "Test automation"),
]

first_names = [
    "Aarav", "Diya", "Ishaan", "Meera", "Kabir", "Ananya",
    "Rohan", "Priya", "Arjun", "Sneha", "Aditya", "Kavya",
    "Rahul", "Nisha", "Vikram", "Pooja", "Karan", "Aisha",
    "Dev", "Neha", "Manav", "Tara", "Varun", "Ira",
    "Sahil", "Riya", "Nikhil", "Maya", "Yash", "Sara",
    "Om", "Anika", "Reyansh", "Sana", "Krish"
]

resume_files = []

for i in range(35):
    role, base_skills, focus = profiles[i % len(profiles)]
    name = first_names[i]
    experience = 1 + (i * 7 % 10)

    skills = list(base_skills)
    extra_skills = ["REST APIs", "Git", "Agile", "Linux", "MongoDB", "Testing"]
    for skill in extra_skills:
        if skill not in skills and random.random() < 0.25:
            skills.append(skill)

    education = random.choice([
        "B.E. in Computer Science",
        "B.Tech in Information Technology",
        "MCA",
        "B.E. in Electronics and Communication",
        "M.Tech in Computer Science"
    ])

    resume_text = f"""Name: {name}
Target Role: {role}
Experience: {experience} years
Education: {education}
Skills: {", ".join(skills)}

Professional Summary:
{ name } is a software professional with {experience} years of experience
in {focus}. Experienced with {", ".join(skills[:3])}.
Interested in building reliable, maintainable software systems.

Experience Details:
Worked on projects involving {", ".join(skills)}.
Collaborated with cross-functional teams, reviewed code, and contributed
to debugging, testing, documentation, and software delivery.
"""

    file_path = RESUME_DIR / f"resume_{i+1:02d}.txt"
    file_path.write_text(resume_text, encoding="utf-8")
    resume_files.append(file_path)

job_descriptions = [
    {
        "job_id": "JD001",
        "title": "Python Backend Engineer",
        "description": "Build backend services and REST APIs using Python, FastAPI or Django, SQL and PostgreSQL. Write tests and maintain production code.",
        "must_have": ["Python"],
        "minimum_experience": 3
    },
    {
        "job_id": "JD002",
        "title": "Java Full Stack Developer",
        "description": "Develop enterprise applications using Java, Spring Boot, Angular, REST APIs and SQL. Collaborate with frontend and backend teams.",
        "must_have": ["Java", "Spring Boot"],
        "minimum_experience": 3
    },
    {
        "job_id": "JD003",
        "title": "Machine Learning Engineer",
        "description": "Build machine learning solutions using Python, scikit-learn and TensorFlow. Work with datasets, SQL and model evaluation.",
        "must_have": ["Python"],
        "minimum_experience": 2
    },
    {
        "job_id": "JD004",
        "title": "Cloud DevOps Engineer",
        "description": "Manage AWS infrastructure, Docker containers, Kubernetes deployments, Linux systems and automated deployment pipelines.",
        "must_have": ["AWS", "Docker"],
        "minimum_experience": 3
    },
    {
        "job_id": "JD005",
        "title": "Frontend Developer",
        "description": "Build responsive user interfaces using JavaScript, React, HTML and CSS. Collaborate with API developers and write maintainable code.",
        "must_have": ["JavaScript", "HTML"],
        "minimum_experience": 2
    }
]

for jd in job_descriptions:
    path = JD_DIR / f"{jd['job_id']}.json"
    path.write_text(json.dumps(jd, indent=2), encoding="utf-8")

print(f"Created {len(resume_files)} resume files.")
print(f"Created {len(job_descriptions)} job descriptions.")
print("\nSample resume:")
print(resume_files[0].read_text(encoding="utf-8"))
print("\nJob description files:")
for path in sorted(JD_DIR.glob("*.json")):
    print("-", path.name)


Created 35 resume files.
Created 5 job descriptions.

Sample resume:
Name: Aarav
Target Role: Software Engineer
Experience: 1 years
Education: M.Tech in Computer Science
Skills: Python, Java, SQL, Git, Agile, MongoDB

Professional Summary:
Aarav is a software professional with 1 years of experience
in Backend development. Experienced with Python, Java, SQL.
Interested in building reliable, maintainable software systems.

Experience Details:
Worked on projects involving Python, Java, SQL, Git, Agile, MongoDB.
Collaborated with cross-functional teams, reviewed code, and contributed
to debugging, testing, documentation, and software delivery.


Job description files:
- JD001.json
- JD002.json
- JD003.json
- JD004.json
- JD005.json


In [5]:

from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

test_embedding = embedding_model.encode(
    "Python developer with experience in REST APIs"
)

print("Model loaded successfully!")
print("Embedding dimensions:", len(test_embedding))


Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Embedding dimensions: 384


In [6]:

%%writefile /content/resume-rag-project/resume_rag.py

from pathlib import Path
import re
import json
import chromadb
from sentence_transformers import SentenceTransformer


PROJECT_DIR = Path("/content/resume-rag-project")
RESUME_DIR = PROJECT_DIR / "resumes"
VECTOR_DB_DIR = PROJECT_DIR / "vector_db"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"


def extract_metadata(text, resume_path):
    """Extract basic candidate metadata from resume text."""
    name_match = re.search(r"(?im)^Name:\s*(.+)$", text)
    exp_match = re.search(r"(?im)^Experience:\s*(\d+)\s*years?", text)
    education_match = re.search(r"(?im)^Education:\s*(.+)$", text)
    skills_match = re.search(r"(?im)^Skills:\s*(.+)$", text)

    return {
        "candidate_name": name_match.group(1).strip() if name_match else "Unknown",
        "experience_years": int(exp_match.group(1)) if exp_match else 0,
        "education": education_match.group(1).strip() if education_match else "Not specified",
        "skills": [
            skill.strip() for skill in skills_match.group(1).split(",")
        ] if skills_match else [],
        "resume_path": str(resume_path),
    }


def split_into_chunks(text, max_chars=900):
    """Split a resume by sections, then split long sections into smaller chunks."""
    sections = re.split(
        r"(?im)(?=^(?:Name|Target Role|Experience|Education|Skills|Professional Summary|Experience Details):)",
        text
    )

    chunks = []

    for section in sections:
        section = section.strip()
        if not section:
            continue

        if len(section) <= max_chars:
            chunks.append(section)
        else:
            # Split long sections at paragraph or sentence boundaries.
            paragraphs = re.split(r"\n+|(?<=[.!?])\s+", section)
            current = ""

            for paragraph in paragraphs:
                if not paragraph.strip():
                    continue

                if len(current) + len(paragraph) + 1 > max_chars and current:
                    chunks.append(current.strip())
                    current = paragraph
                else:
                    current = f"{current}\n{paragraph}".strip()

            if current:
                chunks.append(current.strip())

    return chunks


def build_resume_index(reset=True):
    """Read resume files, embed chunks and save them in ChromaDB."""
    if not RESUME_DIR.exists():
        raise FileNotFoundError(f"Resume directory not found: {RESUME_DIR}")

    resume_paths = sorted(RESUME_DIR.glob("*.txt"))

    if not resume_paths:
        raise ValueError("No .txt resumes found in the resumes folder.")

    VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

    client = chromadb.PersistentClient(path=str(VECTOR_DB_DIR))

    if reset:
        try:
            client.delete_collection("resume_chunks")
        except Exception:
            pass

    collection = client.get_or_create_collection(
        name="resume_chunks",
        metadata={"hnsw:space": "cosine"}
    )

    model = SentenceTransformer(MODEL_NAME)

    ids, documents, metadatas = [], [], []
    total_chunks = 0

    for resume_path in resume_paths:
        text = resume_path.read_text(encoding="utf-8")
        candidate = extract_metadata(text, resume_path)
        chunks = split_into_chunks(text)

        for chunk_number, chunk in enumerate(chunks):
            chunk_id = f"{resume_path.stem}_chunk_{chunk_number}"

            ids.append(chunk_id)
            documents.append(chunk)
            metadatas.append({
                "candidate_name": candidate["candidate_name"],
                "experience_years": candidate["experience_years"],
                "education": candidate["education"],
                "skills": json.dumps(candidate["skills"]),
                "resume_path": str(resume_path),
                "chunk_number": chunk_number,
            })

        total_chunks += len(chunks)

    # Batch embeddings to avoid processing the entire dataset at once.
    batch_size = 64

    for start in range(0, len(documents), batch_size):
        batch_docs = documents[start:start + batch_size]
        batch_ids = ids[start:start + batch_size]
        batch_metadata = metadatas[start:start + batch_size]

        embeddings = model.encode(
            batch_docs,
            normalize_embeddings=True,
            show_progress_bar=False
        ).tolist()

        collection.add(
            ids=batch_ids,
            documents=batch_docs,
            metadatas=batch_metadata,
            embeddings=embeddings
        )

    return {
        "resume_count": len(resume_paths),
        "chunk_count": total_chunks,
        "indexed_chunk_count": collection.count(),
        "collection": collection.name,
        "database_path": str(VECTOR_DB_DIR),
    }


def get_resume_collection():
    """Open the already-built resume collection."""
    client = chromadb.PersistentClient(path=str(VECTOR_DB_DIR))
    return client.get_collection("resume_chunks")


if __name__ == "__main__":
    result = build_resume_index()
    print(json.dumps(result, indent=2))


Writing /content/resume-rag-project/resume_rag.py


In [7]:

import sys
sys.path.insert(0, str(PROJECT_DIR))

from resume_rag import build_resume_index, get_resume_collection

index_result = build_resume_index()

print("\nIndex build result:")
print(index_result)

collection = get_resume_collection()
print("\nTotal searchable chunks:", collection.count())

sample = collection.peek(limit=2)
print("\nSample indexed text:")
for document in sample["documents"]:
    print("-", document[:200].replace("\n", " "))


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Index build result:
{'resume_count': 35, 'chunk_count': 245, 'indexed_chunk_count': 245, 'collection': 'resume_chunks', 'database_path': '/content/resume-rag-project/vector_db'}

Total searchable chunks: 245

Sample indexed text:
- Name: Aarav
- Target Role: Software Engineer


In [8]:

%%writefile /content/resume-rag-project/job_matcher.py

from pathlib import Path
import json
import re
import time

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from resume_rag import get_resume_collection, MODEL_NAME


PROJECT_DIR = Path("/content/resume-rag-project")
JD_DIR = PROJECT_DIR / "job_descriptions"


def tokenize(text):
    """Tokenize text for BM25 keyword retrieval."""
    return re.findall(r"[a-zA-Z0-9+#.]+", text.lower())


def load_job_description(jd_path):
    with open(jd_path, "r", encoding="utf-8") as file:
        return json.load(file)


def match_job(job_description, top_k=10):
    """
    Rank resumes using semantic retrieval, BM25, skills overlap,
    and experience. Enforce required skills and minimum experience.
    """
    start_time = time.perf_counter()

    if isinstance(job_description, (str, Path)):
        job = load_job_description(job_description)
    else:
        job = job_description

    description = job["description"]
    required_skills = [s.lower() for s in job.get("must_have", [])]
    minimum_experience = int(job.get("minimum_experience", 0))

    collection = get_resume_collection()
    count = collection.count()

    if count == 0:
        raise ValueError("Resume index is empty. Build the index first.")

    # Retrieve all chunks so candidates can be ranked and filtered
    # after combining semantic and keyword evidence.
    stored = collection.get(include=["documents", "metadatas"])

    documents = stored["documents"]
    metadatas = stored["metadatas"]
    ids = stored["ids"]

    model = SentenceTransformer(MODEL_NAME)
    query_embedding = model.encode(
        description,
        normalize_embeddings=True
    ).tolist()

    semantic_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=count,
        include=["documents", "metadatas", "distances"]
    )

    semantic_distance = {}
    for doc_id, metadata, distance in zip(
        semantic_results["ids"][0],
        semantic_results["metadatas"][0],
        semantic_results["distances"][0]
    ):
        # Chroma's cosine distance is converted to a 0-100 similarity.
        similarity = max(0.0, min(100.0, (1.0 - float(distance)) * 100))
        path = metadata["resume_path"]
        semantic_distance[path] = max(
            semantic_distance.get(path, 0.0), similarity
        )

    # BM25 scores are calculated over indexed resume chunks.
    tokenized_documents = [tokenize(doc) for doc in documents]
    bm25 = BM25Okapi(tokenized_documents)
    bm25_scores = bm25.get_scores(tokenize(description))

    bm25_by_resume = {}
    for doc_id, metadata, score in zip(ids, metadatas, bm25_scores):
        path = metadata["resume_path"]
        bm25_by_resume[path] = max(
            bm25_by_resume.get(path, 0.0), float(score)
        )

    max_bm25 = max(bm25_by_resume.values(), default=0.0)

    # Group the indexed chunks by candidate resume.
    candidates = {}

    for metadata in metadatas:
        path = metadata["resume_path"]
        if path not in candidates:
            candidates[path] = {
                "candidate_name": metadata["candidate_name"],
                "experience_years": int(metadata["experience_years"]),
                "education": metadata["education"],
                "skills": json.loads(metadata["skills"]),
                "resume_path": path,
                "chunks": []
            }

    for doc, metadata in zip(documents, metadatas):
        candidates[metadata["resume_path"]]["chunks"].append(doc)

    ranked = []

    for path, candidate in candidates.items():
        candidate_skills = [
            skill.lower() for skill in candidate["skills"]
        ]

        matched_skills = [
            skill for skill in candidate["skills"]
            if skill.lower() in description.lower()
        ]

        # A skill can be required even when the job description's
        # wording varies, so check the must-have list separately.
        missing_required = [
            skill for skill in required_skills
            if skill not in candidate_skills
        ]

        eligible = (
            not missing_required
            and candidate["experience_years"] >= minimum_experience
        )

        if not eligible:
            continue

        semantic_score = semantic_distance.get(path, 0.0)
        lexical_score = (
            bm25_by_resume.get(path, 0.0) / max_bm25 * 100
            if max_bm25 > 0 else 0.0
        )

        # Skills overlap is measured against job-mentioned skills.
        job_skill_terms = set(tokenize(description))
        candidate_skill_terms = set(tokenize(" ".join(candidate["skills"])))
        relevant_terms = job_skill_terms.intersection(candidate_skill_terms)
        skills_score = (
            len(relevant_terms) / max(len(candidate_skill_terms), 1) * 100
        )

        experience_score = min(
            100.0,
            candidate["experience_years"] /
            max(minimum_experience, 1) * 100
        )

        # Heuristic score, not an objective measure of hiring suitability.
        match_score = (
            0.40 * semantic_score
            + 0.25 * lexical_score
            + 0.25 * skills_score
            + 0.10 * experience_score
        )

        excerpts = [
            chunk for chunk in candidate["chunks"]
            if any(skill.lower() in chunk.lower()
                   for skill in matched_skills)
        ]

        if not excerpts:
            excerpts = candidate["chunks"][:2]

        reasoning = (
            f"Meets the minimum experience requirement of "
            f"{minimum_experience} years and has all required skills. "
            f"Semantic score: {semantic_score:.1f}/100; "
            f"keyword score: {lexical_score:.1f}/100."
        )

        ranked.append({
            "candidate_name": candidate["candidate_name"],
            "resume_path": candidate["resume_path"],
            "match_score": round(match_score, 2),
            "matched_skills": matched_skills,
            "relevant_excerpts": excerpts[:3],
            "reasoning": reasoning,
            "experience_years": candidate["experience_years"],
            "education": candidate["education"],
            "semantic_score": round(semantic_score, 2),
            "keyword_score": round(lexical_score, 2),
            "eligible": True
        })

    ranked.sort(key=lambda item: item["match_score"], reverse=True)

    elapsed_ms = round((time.perf_counter() - start_time) * 1000, 2)

    return {
        "job_description": {
            "job_id": job.get("job_id", "CUSTOM"),
            "title": job.get("title", "Untitled job"),
            "description": description,
            "must_have": job.get("must_have", []),
            "minimum_experience": minimum_experience
        },
        "total_candidates_indexed": len(candidates),
        "eligible_candidates": len(ranked),
        "top_matches": ranked[:top_k],
        "latency_ms": elapsed_ms,
        "scoring_note": (
            "Scores are heuristic ranking estimates, not probabilities "
            "of job success or objective hiring suitability."
        )
    }


def match_job_file(job_file, top_k=10, output_file=None):
    result = match_job(job_file, top_k=top_k)

    if output_file:
        output_path = Path(output_file)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        output_path.write_text(
            json.dumps(result, indent=2),
            encoding="utf-8"
        )

    return result


if __name__ == "__main__":
    job_files = sorted(JD_DIR.glob("*.json"))

    if not job_files:
        raise FileNotFoundError("No job description JSON files found.")

    result = match_job_file(
        job_files[0],
        top_k=10,
        output_file=PROJECT_DIR / "evaluation" / "sample_matches.json"
    )

    print(json.dumps(result, indent=2))


Writing /content/resume-rag-project/job_matcher.py


In [9]:

import sys
import json

sys.path.insert(0, str(PROJECT_DIR))

from job_matcher import match_job_file

job_file = PROJECT_DIR / "job_descriptions" / "JD001.json"
output_file = PROJECT_DIR / "evaluation" / "sample_matches.json"

result = match_job_file(
    job_file,
    top_k=10,
    output_file=output_file
)

print("Job:", result["job_description"]["title"])
print("Resumes indexed:", result["total_candidates_indexed"])
print("Eligible candidates:", result["eligible_candidates"])
print("Latency (ms):", result["latency_ms"])
print("\nTop matches:")

for candidate in result["top_matches"]:
    print(
        f"\n{candidate['candidate_name']} | "
        f"Score: {candidate['match_score']}/100"
    )
    print("Skills:", ", ".join(candidate["matched_skills"]))
    print("Reason:", candidate["reasoning"])

print("\nJSON saved to:", output_file)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Job: Python Backend Engineer
Resumes indexed: 35
Eligible candidates: 10
Latency (ms): 2082.64

Top matches:

Priya | Score: 77.89/100
Skills: Python, Django, FastAPI, PostgreSQL
Reason: Meets the minimum experience requirement of 3 years and has all required skills. Semantic score: 69.7/100; keyword score: 100.0/100.

Aisha | Score: 77.89/100
Skills: Python, Django, FastAPI, PostgreSQL
Reason: Meets the minimum experience requirement of 3 years and has all required skills. Semantic score: 69.7/100; keyword score: 100.0/100.

Maya | Score: 73.58/100
Skills: Python, Django, FastAPI, PostgreSQL
Reason: Meets the minimum experience requirement of 3 years and has all required skills. Semantic score: 68.3/100; keyword score: 95.1/100.

Sneha | Score: 72.54/100
Skills: Python, SQL, REST APIs
Reason: Meets the minimum experience requirement of 3 years and has all required skills. Semantic score: 62.8/100; keyword score: 83.0/100.

Sara | Score: 68.73/100
Skills: Python, SQL, REST APIs
Reason:

In [10]:

import sys
import json
import time
import pandas as pd

sys.path.insert(0, str(PROJECT_DIR))

from job_matcher import match_job_file
from resume_rag import extract_metadata

RESUME_DIR = PROJECT_DIR / "resumes"
JD_DIR = PROJECT_DIR / "job_descriptions"
EVAL_DIR = PROJECT_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# Build a simple ground-truth set from the synthetic data.
# A resume is relevant when it meets the JD's must-have skills,
# minimum experience, and has at least one skill mentioned in the JD.
resume_records = []

for path in sorted(RESUME_DIR.glob("*.txt")):
    text = path.read_text(encoding="utf-8")
    metadata = extract_metadata(text, path)
    resume_records.append(metadata)

evaluation_rows = []
all_results = []

for jd_path in sorted(JD_DIR.glob("*.json")):
    job = json.loads(jd_path.read_text(encoding="utf-8"))
    jd_text = job["description"].lower()
    must_have = [s.lower() for s in job.get("must_have", [])]
    min_exp = int(job.get("minimum_experience", 0))

    relevant_paths = set()

    for resume in resume_records:
        candidate_skills = [s.lower() for s in resume["skills"]]

        meets_must_have = all(
            skill in candidate_skills for skill in must_have
        )
        meets_experience = resume["experience_years"] >= min_exp
        has_relevant_skill = any(
            skill in jd_text for skill in candidate_skills
        )

        if meets_must_have and meets_experience and has_relevant_skill:
            relevant_paths.add(resume["resume_path"])

    started = time.perf_counter()

    result = match_job_file(
        jd_path,
        top_k=10,
        output_file=EVAL_DIR / f"{job['job_id']}_matches.json"
    )

    measured_latency_ms = (time.perf_counter() - started) * 1000

    returned = result["top_matches"]
    returned_paths = {candidate["resume_path"] for candidate in returned}
    relevant_returned = returned_paths.intersection(relevant_paths)

    precision_at_10 = (
        len(relevant_returned) / len(returned) if returned else 0.0
    )
    recall_at_10 = (
        len(relevant_returned) / len(relevant_paths)
        if relevant_paths else 0.0
    )

    row = {
        "job_id": job["job_id"],
        "job_title": job["title"],
        "eligible_candidates": result["eligible_candidates"],
        "ground_truth_relevant": len(relevant_paths),
        "returned_candidates": len(returned),
        "precision_at_10": round(precision_at_10, 4),
        "recall_at_10": round(recall_at_10, 4),
        "latency_ms": round(measured_latency_ms, 2)
    }

    evaluation_rows.append(row)
    all_results.append(result)

    print(f"\n{job['job_id']}: {job['title']}")
    print(f"Relevant candidates: {len(relevant_paths)}")
    print(f"Returned candidates: {len(returned)}")
    print(f"Precision@10: {precision_at_10:.2%}")
    print(f"Recall@10: {recall_at_10:.2%}")
    print(f"Measured latency: {measured_latency_ms:.2f} ms")

evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df.to_csv(EVAL_DIR / "evaluation_metrics.csv", index=False)

summary = {
    "job_descriptions_evaluated": len(evaluation_rows),
    "mean_precision_at_10": round(
        evaluation_df["precision_at_10"].mean(), 4
    ),
    "mean_recall_at_10": round(
        evaluation_df["recall_at_10"].mean(), 4
    ),
    "mean_latency_ms": round(
        evaluation_df["latency_ms"].mean(), 2
    ),
    "evaluation_note": (
        "Rule-based relevance labels on synthetic resumes; "
        "not a measure of real-world hiring quality."
    )
}

(EVAL_DIR / "evaluation_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("\n=== Evaluation table ===")
display(evaluation_df)

print("\n=== Summary ===")
print(json.dumps(summary, indent=2))
print("\nSaved metrics to:", EVAL_DIR / "evaluation_metrics.csv")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


JD001: Python Backend Engineer
Relevant candidates: 10
Returned candidates: 10
Precision@10: 100.00%
Recall@10: 100.00%
Measured latency: 1358.87 ms


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


JD002: Java Full Stack Developer
Relevant candidates: 7
Returned candidates: 7
Precision@10: 100.00%
Recall@10: 100.00%
Measured latency: 1001.32 ms


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...


JD004: Cloud DevOps Engineer
Relevant candidates: 6
Returned candidates: 6
Precision@10: 100.00%
Recall@10: 100.00%
Measured latency: 1026.89 ms


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


JD005: Frontend Developer
Relevant candidates: 4
Returned candidates: 4
Precision@10: 100.00%
Recall@10: 100.00%
Measured latency: 1045.76 ms

=== Evaluation table ===


,job_id,job_title,eligible_candidates,ground_truth_relevant,returned_candidates,precision_at_10,recall_at_10,latency_ms
0,JD001,Python Backend Engineer,10,10,10,1.0,1.0000,1358.87
1,JD002,Java Full Stack Developer,7,7,7,1.0,1.0000,1001.32
2,JD003,Machine Learning Engineer,14,14,10,1.0,0.7143,1025.39
3,JD004,Cloud DevOps Engineer,6,6,6,1.0,1.0000,1026.89
4,JD005,Frontend Developer,4,4,4,1.0,1.0000,1045.76



=== Summary ===
{
  "job_descriptions_evaluated": 5,
  "mean_precision_at_10": 1.0,
  "mean_recall_at_10": 0.9429,
  "mean_latency_ms": 1091.65,
  "evaluation_note": "Rule-based relevance labels on synthetic resumes; not a measure of real-world hiring quality."
}

Saved metrics to: /content/resume-rag-project/evaluation/evaluation_metrics.csv


# Resume RAG and Job Matching — Experiments and Analysis

## 1. Objective

The objective of this experiment is to evaluate a resume retrieval and job-matching pipeline using semantic embeddings, ChromaDB vector search, BM25 keyword retrieval, and heuristic candidate ranking.

## 2. Dataset

- 35 synthetic resume files.
- 5 job descriptions covering Python backend development, Java full stack development, machine learning, DevOps, and frontend development.
- Resume metadata includes candidate name, skills, experience, education, and file path.

## 3. Methodology

1. Split resume text into smaller, section-aware chunks.
2. Generate embeddings using `sentence-transformers/all-MiniLM-L6-v2`.
3. Store chunks, embeddings, and metadata in ChromaDB.
4. Retrieve semantically relevant content and calculate BM25 keyword scores.
5. Combine semantic similarity, keyword relevance, skills overlap, and experience into a heuristic match score.
6. Apply must-have skill and minimum-experience filters.
7. Evaluate candidate retrieval using Precision@10, Recall@10, and measured latency.

## 4. Evaluation Metrics

- **Precision@10:** Fraction of the returned top-10 candidates meeting the defined relevance criteria.
- **Recall@10:** Fraction of all candidates meeting those criteria that appear in the returned top 10.
- **Latency:** Measured time taken by the job-matching pipeline.

## 5. Limitations

The dataset is synthetic, and relevance labels are generated using predefined rules. The metrics therefore evaluate this experimental setup, not real-world hiring success. The scoring formula is heuristic and should not be interpreted as a probability of job performance.

## 6. Results and Discussion

The results are generated by the evaluation code and saved in `evaluation/evaluation_metrics.csv` and `evaluation/evaluation_summary.json`. Actual measured values should be reported here after the evaluation has run.

## 7. Future Improvements

- Test with a larger, more diverse dataset.
- Add PDF and DOCX resume ingestion.
- Compare dense-only, BM25-only, and hybrid retrieval.
- Tune chunk size and retrieval weights.
- Add manually reviewed relevance labels.
- Improve skills normalization and must-have matching.


In [11]:

import json
import pandas as pd
from IPython.display import display

metrics_path = PROJECT_DIR / "evaluation" / "evaluation_metrics.csv"
summary_path = PROJECT_DIR / "evaluation" / "evaluation_summary.json"

evaluation_df = pd.read_csv(metrics_path)
summary = json.loads(summary_path.read_text(encoding="utf-8"))

print("Per-job evaluation results")
display(evaluation_df)

print("\nOverall evaluation summary")
display(pd.DataFrame([summary]))


Per-job evaluation results


,job_id,job_title,eligible_candidates,ground_truth_relevant,returned_candidates,precision_at_10,recall_at_10,latency_ms
0,JD001,Python Backend Engineer,10,10,10,1.0,1.0000,1358.87
1,JD002,Java Full Stack Developer,7,7,7,1.0,1.0000,1001.32
2,JD003,Machine Learning Engineer,14,14,10,1.0,0.7143,1025.39
3,JD004,Cloud DevOps Engineer,6,6,6,1.0,1.0000,1026.89
4,JD005,Frontend Developer,4,4,4,1.0,1.0000,1045.76



Overall evaluation summary


,job_descriptions_evaluated,mean_precision_at_10,mean_recall_at_10,mean_latency_ms,evaluation_note
0,5,1.0,0.9429,1091.65,Rule-based relevance labels on synthetic resum...
